# 02 · Data quality analysis
**Step 4:** missing values, duplicates, incorrect types, inconsistent/invalid values, outliers
and primary-key uniqueness. This notebook only *inspects*; nothing is changed until notebook 03.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

# Where the raw CSVs live. To rehearse on generated test data instead, use:
#   RAW_DIR = config.SYNTHETIC_DIR   (run `python scripts/make_synthetic_data.py` first)
RAW_DIR = config.RAW_DIR

In [ ]:
from src.data_loader import load_raw_tables
from src.quality import run_quality_report, quality_summary

raw = load_raw_tables(RAW_DIR)
report = run_quality_report(raw)      # also saved to reports/data_quality_report.json
display(quality_summary(report))

## Missing values

In [ ]:
miss = pd.DataFrame([(t, c, n, r["missing_pct"][c]) for t, r in report.items()
                     for c, n in r["missing_values"].items()],
                    columns=["table", "column", "missing", "pct"]).sort_values("pct", ascending=False)
display(miss)
if len(miss):
    top = miss.head(15).assign(label=lambda d: d["table"] + "." + d["column"])
    sns.barplot(data=top, x="pct", y="label", color="#1b7f5c")
    plt.xlabel("% missing"); plt.ylabel(""); plt.title("Columns with the most missing values")
    plt.show()

## Duplicates and primary-key uniqueness

In [ ]:
display(pd.DataFrame([{"table": t, "primary_key": ", ".join(r["primary_key"]),
                       "duplicate_rows": r["duplicate_rows"],
                       "pk_duplicates": r["primary_key_duplicates"],
                       "pk_unique": r["primary_key_unique"]} for t, r in report.items()]))

## Incorrect data types and invalid values

In [ ]:
for t, r in report.items():
    print(f"\n--- {t} ---")
    print("dtypes:", r["dtypes"])
    print("invalid values:", r["invalid_values"] or "none found")

## Inconsistent categorical values and outliers (IQR rule)

In [ ]:
for t, r in report.items():
    print(f"\n--- {t} ---")
    print("text columns whose spellings collapse after trim/case/accent normalisation:",
          r["inconsistent_categorical_variants"] or "none")
    flagged = {c: n for c, n in r["iqr_outliers"].items() if n}
    print("IQR outlier counts:", flagged or "none")

### Reading the results
Record here which issues you found and the handling you choose in notebook 03
(drop, impute, correct, or keep and flag).